# 📊 Notebook 03 — Exploratory Data Analysis (EDA)
**Healthcare RAG-Powered Medical Q&A Assistant**

**Goal:** Understand the dataset visually before building any model.

We will answer questions like:
- What categories appear most?
- How long are the questions and answers?
- What medical words appear most often?
- Do all records have context?

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter

# Optional: word cloud
try:
    from wordcloud import WordCloud, STOPWORDS
    wordcloud_available = True
except ImportError:
    wordcloud_available = False
    print('Note: wordcloud not installed. Run: pip install wordcloud')

# Make charts look nice
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Folder to save charts
os.makedirs('../reports', exist_ok=True)

print('Ready!')

## Step 2 — Load the Cleaned Data

In [ ]:
df = pd.read_csv('../data/processed/pubmedqa_cleaned.csv')

print(f'Rows: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(3)

## Chart 1 — How many records per category?

In [ ]:
category_counts = df['category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart
axes[0].bar(category_counts.index, category_counts.values,
            color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860'])
axes[0].set_title('Number of Records per Category', fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')

# Add count labels on top of each bar
for i, count in enumerate(category_counts.values):
    axes[0].text(i, count + 30, str(count), ha='center', fontsize=9)

# Right: pie chart
axes[1].pie(category_counts.values,
            labels=category_counts.index,
            autopct='%1.1f%%',
            startangle=90)
axes[1].set_title('Category Share', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/01_category_distribution.png')
plt.show()

print('Most common category:', category_counts.index[0])

## Chart 2 — How long are the questions?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: question word count histogram
axes[0].hist(df['question_word_count'], bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('Question Length (words)', fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
# Red line = median
median_q = df['question_word_count'].median()
axes[0].axvline(median_q, color='red', linestyle='--', label=f'Median: {median_q:.0f} words')
axes[0].legend()

# Right: boxplot per category
order = df['category'].value_counts().index.tolist()
sns.boxplot(data=df, x='category', y='question_word_count',
            order=order, ax=axes[1], showfliers=False)
axes[1].set_title('Question Length by Category', fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('../reports/02_question_length.png')
plt.show()

print(f'Average question length: {df["question_word_count"].mean():.1f} words')
print(f'Median  question length: {df["question_word_count"].median():.0f} words')

## Chart 3 — How long are the answers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clip extreme outliers for cleaner chart (keep 99% of data)
clip_val = df['answer_word_count'].quantile(0.99)

# Left: answer word count histogram
axes[0].hist(df['answer_word_count'].clip(upper=clip_val),
             bins=40, color='#55A868', edgecolor='white')
axes[0].set_title('Answer Length (words)', fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
median_a = df['answer_word_count'].median()
axes[0].axvline(median_a, color='red', linestyle='--', label=f'Median: {median_a:.0f} words')
axes[0].legend()

# Right: boxplot per category
sns.boxplot(data=df, x='category', y='answer_word_count',
            order=order, ax=axes[1], showfliers=False)
axes[1].set_title('Answer Length by Category', fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('../reports/03_answer_length.png')
plt.show()

print(f'Average answer length: {df["answer_word_count"].mean():.1f} words')
print(f'Median  answer length: {df["answer_word_count"].median():.0f} words')

## Chart 4 — Do all records have a context?

In [ ]:
# A record has context if the context column is not empty
has_context = df['context'].str.strip().str.len() > 0

with_ctx    = has_context.sum()
without_ctx = len(df) - with_ctx

fig, ax = plt.subplots(figsize=(6, 5))
ax.pie([with_ctx, without_ctx],
       labels=['Has Context', 'No Context'],
       autopct='%1.1f%%',
       colors=['#4C72B0', '#C44E52'],
       startangle=90)
ax.set_title('Context Availability', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/04_context_coverage.png')
plt.show()

print(f'Records with context   : {with_ctx:,} ({with_ctx/len(df)*100:.1f}%)')
print(f'Records without context: {without_ctx:,} ({without_ctx/len(df)*100:.1f}%)')

## Chart 5 — What words appear most in questions?

In [ ]:
# Split all questions into individual words and count them
all_words = ' '.join(df['question'].tolist()).lower().split()

# Remove common words that don't carry meaning
stopwords = {'the','a','an','is','are','was','were','in','of','to','and',
             'for','with','on','at','by','or','be','this','that','it',
             'as','from','have','has','not','does','do','what','which',
             'can','will','more','than','its'}

filtered_words = [w for w in all_words if w not in stopwords and len(w) > 3]
top_30 = Counter(filtered_words).most_common(30)

words, counts = zip(*top_30)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(list(words)[::-1], list(counts)[::-1], color='#4C72B0')
ax.set_title('Top 30 Most Frequent Words in Questions', fontweight='bold')
ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig('../reports/05_top_words.png')
plt.show()

print('Top 10 words:', list(words[:10]))

## Chart 6 — Word Cloud (bonus visual)

In [ ]:
if wordcloud_available:
    text = ' '.join(df['question'].tolist())

    # Extra words to remove from the cloud
    extra_stops = set(STOPWORDS)
    extra_stops.update(['patients','study','based','using','effect',
                        'compared','whether','clinical','data','group'])

    wc = WordCloud(
        width=900,
        height=400,
        background_color='white',
        stopwords=extra_stops,
        colormap='Blues',
        max_words=100
    ).generate(text)

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.imshow(wc)
    ax.axis('off')
    ax.set_title('Word Cloud — Medical Questions', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('../reports/06_wordcloud.png')
    plt.show()
else:
    print('Install wordcloud to see this chart: pip install wordcloud')

## Summary — Key Findings

In [ ]:
print('========== EDA SUMMARY ==========')
print(f'Total records             : {len(df):,}')
print(f'Number of categories      : {df["category"].nunique()}')
print(f'Most common category      : {df["category"].value_counts().index[0]}')
print(f'Avg question length       : {df["question_word_count"].mean():.1f} words')
print(f'Avg answer length         : {df["answer_word_count"].mean():.1f} words')
print(f'Records with context      : {has_context.sum():,} ({has_context.mean()*100:.1f}%)')
print('=================================')
print()
print('Charts saved to reports/ folder')

## ✅ Done!

| Chart | What it shows |
|---|---|
| Chart 1 | How many records per medical category |
| Chart 2 | How long questions are |
| Chart 3 | How long answers are |
| Chart 4 | Whether records have context |
| Chart 5 | Most frequent words in questions |
| Chart 6 | Word cloud of medical terms |

**➡️ Next: Open `04_embeddings_vectorstore.ipynb`**